# OOP Week 9 -- Factory & Registry Pattern

**Course:** Object-Oriented Programming (Year 2)
**Session:** 3 hours
**Prerequisites:** Weeks 1-8
**Focus:** creating objects from config, plugin architecture

---

## Learning Objectives

1. Explain the Factory Pattern and when to use it
2. Build a Registry that maps names to classes
3. Create objects from configuration dictionaries
4. Understand how this enables plugin architecture
5. Register new components without modifying existing code

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Section 1: What is a Factory?

A **Factory** is an object (or function) that **creates other objects**. Instead of using `ClassName(...)` directly, you ask the factory to build it for you.

### The Pizza Analogy

Instead of making pizza yourself, you tell the pizza shop: 'I want a Margherita.' The shop (factory) knows how to make it. Tomorrow they can add a new pizza type without changing how you order.

### In Our Pipeline

We want to create analyzers from a config file:
```json
{"analyzers": ["mean", "std", {"name": "events", "threshold": 50}]}
```
The factory reads this config and creates the right objects.

### The AnalyzerBase (from previous weeks)

In [ ]:
class AnalyzerBase:
    """Base class for analyzers."""
    def analyze(self, values):
        raise NotImplementedError

class MeanAnalyzer(AnalyzerBase):
    def analyze(self, values):
        return {"mean": round(sum(values)/len(values), 4)} if values else {}

class StdAnalyzer(AnalyzerBase):
    def analyze(self, values):
        if not values:
            return {}
        m = sum(values) / len(values)
        return {"std": round((sum((x-m)**2 for x in values)/len(values))**0.5, 4)}

class EventAnalyzer(AnalyzerBase):
    def __init__(self, threshold=50):
        self.threshold = threshold
    def analyze(self, values):
        return {"events_above": sum(1 for v in values if v > self.threshold)}

print("Analyzer classes defined.")

**Expected Output:**
```
Analyzer classes defined.
```

---
## Section 2: The Registry Pattern

In [ ]:
class AnalyzerFactory:
    """Factory + Registry for creating analyzers from config."""

    _registry = {}

    @classmethod
    def register(cls, name, analyzer_class):
        """Register an analyzer class under a name."""
        cls._registry[name] = analyzer_class
        print("Registered: " + name + " -> " + analyzer_class.__name__)

    @classmethod
    def create(cls, name, **kwargs):
        """Create an analyzer by name."""
        if name not in cls._registry:
            available = list(cls._registry.keys())
            raise ValueError("Unknown analyzer: " + name
                           + ". Available: " + str(available))
        return cls._registry[name](**kwargs)

    @classmethod
    def list_available(cls):
        """List all registered analyzers."""
        return list(cls._registry.keys())


# Register built-in analyzers
AnalyzerFactory.register("mean", MeanAnalyzer)
AnalyzerFactory.register("std", StdAnalyzer)
AnalyzerFactory.register("events", EventAnalyzer)

print("Available:", AnalyzerFactory.list_available())

**Expected Output:**
```
Registered: mean -> MeanAnalyzer
Registered: std -> StdAnalyzer
Registered: events -> EventAnalyzer
Available: ['mean', 'std', 'events']
```

---
## Section 3: Creating Objects from Config

In [ ]:
# Config (could be loaded from JSON/YAML)
config = {
    "analyzers": [
        {"name": "mean"},
        {"name": "std"},
        {"name": "events", "params": {"threshold": 40}},
    ]
}

# Build analyzers from config
analyzers = []
for spec in config["analyzers"]:
    params = spec.get("params", {})
    a = AnalyzerFactory.create(spec["name"], **params)
    analyzers.append(a)
    print("Created: " + spec["name"] + " -> " + type(a).__name__)

# Run all
values = [10, 25, 30, 55, 20, 45, 60]
combined = {}
for a in analyzers:
    combined.update(a.analyze(values))

print()
print("Results:", combined)

**Expected Output:**
```
Created: mean -> MeanAnalyzer
Created: std -> StdAnalyzer
Created: events -> EventAnalyzer

Results: {'mean': 35.0, 'std': 16.5831, 'events_above': 3}
```

---
## Section 4: Plugin Architecture

The factory + registry enables a **plugin** architecture. Anyone can add new analyzers without modifying existing code -- just create a new class and register it.

In [ ]:
# A plugin: new analyzer, no changes to existing code!
class PercentileAnalyzer(AnalyzerBase):
    def __init__(self, percentiles=None):
        self.percentiles = percentiles or [25, 50, 75]
    def analyze(self, values):
        if not values:
            return {}
        s = sorted(values)
        n = len(s)
        result = {}
        for p in self.percentiles:
            idx = min(int(n * p / 100), n - 1)
            result["p" + str(p)] = s[idx]
        return result

# One line to register!
AnalyzerFactory.register("percentile", PercentileAnalyzer)

# Now it works with the existing pipeline
p = AnalyzerFactory.create("percentile", percentiles=[10, 50, 90])
print("Percentile results:", p.analyze(values))
print()
print("Available analyzers:", AnalyzerFactory.list_available())

**Expected Output:**
```
Registered: percentile -> PercentileAnalyzer
Percentile results: {'p10': 10, 'p50': 30, 'p90': 55}

Available analyzers: ['mean', 'std', 'events', 'percentile']
```

---
### Try It!

Create a `VarianceAnalyzer(AnalyzerBase)`, register it with the factory, then create it from config and run it.

In [ ]:
# YOUR CODE HERE


---
## Section 5: Registry with Metadata

In [ ]:
class EnhancedFactory:
    """Factory that stores metadata about registered classes."""

    _registry = {}

    @classmethod
    def register(cls, name, klass, description="", author=""):
        cls._registry[name] = {
            "class": klass,
            "description": description,
            "author": author,
        }

    @classmethod
    def create(cls, name, **kwargs):
        if name not in cls._registry:
            raise ValueError("Unknown: " + name)
        return cls._registry[name]["class"](**kwargs)

    @classmethod
    def describe(cls):
        for name, info in cls._registry.items():
            print(name + " (" + info["class"].__name__ + ")")
            if info["description"]:
                print("  " + info["description"])


EnhancedFactory.register("mean", MeanAnalyzer,
    description="Computes arithmetic mean", author="core")
EnhancedFactory.register("std", StdAnalyzer,
    description="Computes standard deviation", author="core")
EnhancedFactory.register("events", EventAnalyzer,
    description="Counts events above threshold", author="core")

print("=== Registered Analyzers ===")
EnhancedFactory.describe()

**Expected Output:**
```
=== Registered Analyzers ===
mean (MeanAnalyzer)
  Computes arithmetic mean
std (StdAnalyzer)
  Computes standard deviation
events (EventAnalyzer)
  Counts events above threshold
```

---
## Section 6: Decorator-Based Registration

In [ ]:
# Advanced: register using a decorator
class DecoratorFactory:
    _registry = {}

    @classmethod
    def register(cls, name):
        """Decorator that registers a class."""
        def decorator(klass):
            cls._registry[name] = klass
            return klass
        return decorator

    @classmethod
    def create(cls, name, **kw):
        return cls._registry[name](**kw)


# Usage: just add the decorator!
@DecoratorFactory.register("quick_mean")
class QuickMeanAnalyzer:
    def analyze(self, values):
        return {"mean": sum(values)/len(values)} if values else {}


@DecoratorFactory.register("quick_count")
class QuickCountAnalyzer:
    def analyze(self, values):
        return {"count": len(values)}


# No explicit register call needed!
print("Registered:", list(DecoratorFactory._registry.keys()))
a = DecoratorFactory.create("quick_mean")
print("Result:", a.analyze([10, 20, 30]))

**Expected Output:**
```
Registered: ['quick_mean', 'quick_count']
Result: {'mean': 20.0}
```

---
### Try It!

Create a `CleanerFactory` that works the same way as AnalyzerFactory but for cleaning strategies. Register `DropMissing`, `DropOutOfRange`, and `DropDuplicates`.

In [ ]:
# YOUR CODE HERE


---
### Common Mistake: Factory returning class instead of instance

The code below has a bug. Can you spot it before reading the fix?

In [ ]:
class BadFactory:
    _registry = {"mean": MeanAnalyzer}

    @classmethod
    def create(cls, name):
        return cls._registry[name]  # BUG: returns the CLASS, not an instance!

result = BadFactory.create("mean")
print(type(result))  # <class 'type'> -- it's the class itself!
# result.analyze([1,2,3])  # TypeError: missing self!

**What goes wrong:** The registry stores CLASSES, not instances. You must CALL the class (with parentheses and arguments) to create an instance. `cls._registry[name]` gives you the class; `cls._registry[name](**kw)` gives you an instance.

**The fix:**

In [ ]:
class GoodFactory:
    _registry = {"mean": MeanAnalyzer}

    @classmethod
    def create(cls, name, **kw):
        return cls._registry[name](**kw)  # FIXED: call the class!

result = GoodFactory.create("mean")
print(type(result))  # MeanAnalyzer instance
print(result.analyze([1, 2, 3]))

---
## Build from Scratch Exercise

This exercise tests whether you truly understand this week's concepts. Complete it without looking at the examples above.

In [ ]:
# BUILD FROM SCRATCH:
# Build a ShapeFactory that creates Circle, Rectangle, and Triangle from a config dict. Register each shape. Create shapes from: [{"type": "circle", "radius": 5}, ...]

# YOUR CODE HERE


In [ ]:
# TEST your build-from-scratch code:

# YOUR TESTS HERE


---
## Connect the Dots

How does this week's concept connect to previous weeks?

In [ ]:
# How does the Factory Pattern build on the Strategy Pattern (Week 8)?

# YOUR ANSWER (as comments or code):


---
## Real-World Spotting

OOP patterns are everywhere in real software. Can you spot them?

In [ ]:
# Think about a restaurant ordering system. How would a Factory pattern help create different meal objects from a menu?

# YOUR ANSWER:


---
## Diagram It

Draw an ASCII class diagram for the main classes from this week. Include:
- Class names
- Key attributes
- Key methods
- Relationships (has-a, is-a)

In [ ]:
# Draw your ASCII diagram here:
# +------------------+
# |   ClassName      |
# +------------------+
# | - attribute      |
# +------------------+
# | + method()       |
# +------------------+

# YOUR DIAGRAM:


---
## Key Vocabulary

| Term | Definition |
|------|------------|
| **Factory** | An object that creates other objects |
| **Registry** | A mapping from names to classes |
| **Plugin** | A new component added without modifying core code |
| **`@classmethod`** | A method that belongs to the class, not instances |
| **Decorator** | A function/class that wraps another to add behavior |

---
## Recap Exercise

Without looking at the code above, try to:

In [ ]:
# 1. Write one class from this week FROM MEMORY
#    (it does not need to be perfect)

# YOUR CODE HERE


# 2. Create an instance and call at least one method

# YOUR CODE HERE


# 3. Write one test for your class

# YOUR CODE HERE


---
## What to Review Before Next Week

Before the next session, make sure you can:

1. Explain this week's main concept in your own words
2. Write a simple example from memory
3. Identify this pattern in existing code
4. Explain WHY this pattern is useful (not just HOW)

---
## Mini-Quiz

In [ ]:
# Q1: What is a Factory?
# Answer: 

# Q2: What is a Registry?
# Answer: 

# Q3: How does the factory enable plugin architecture?
# Answer: 

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)